```
┌───────────────────────────────────────────────────────────────────────┐
│ Mini-Projeto Avaliativo - Módulo 2                                    │
│ Aluno: Luiz Felipe F. V. Vieira                                       │
│ "Dashboard Analítico de Compras Públicas em Saúde (BPS- 2020 a 2026)" │
└───────────────────────────────────────────────────────────────────────┘
```
```
┌───────────────────────────────────────────────────────────────────────┐
│ Fluxo do Pipeline de dados:                                           │
│                                                                       │
│ 1- EXTRACT - Verificação e conhecimento da base                       │
│    (Arquitetura Medalão - Camada Bronze)                              │
│                                                                       │
│ 2- TRANSFORM - Auditoria, Limpeza e Tratamento dos Dados              │
│   2.1 -> Arquitetura Medalhão (Camada Prata)                          │
│       2.1.1 -> Concatenação das bases (df_prata)                      │
│       2.1.2 -> Verificação de Nulos e Duplicados                      │
│       2.1.3 -> Verificação de Inconsistência de Domínio               │
│       2.1.4 -> Tratamento, Limpeza e Enriquecimento dos Dados         │
│       2.1.5 -> Otimização de Tipos (Downcasting)                      │
│   2.2 -> Arquitetura Medalhão (Camada Ouro)                           │
│       2.2.1 -> Seleção de Colunas e Engenharia de Features            │
│                (Tabela Única Denormalizada)                           │
│       2.2.2 -> Verificação da Qualidade dos Dados Quantitativos       │
│                (Criação da Flag 'flag_qualidade_dado')                │
│       2.2.3 -> Faixa de Dispersão de Preços dos Produtos              │
│                (Criação de 'coeficiente_variacao' - numérico          │
│                 e da 'faixa_dispersao_preco' - categórica)            │
│       2.2.4 -> Criação de Star Schema                                 │
│                (Tabela Fato e Dimensões)                              │
│                                                                       │
│ 3- LOAD - Exportar DataFrames para novas bases (arquivos)             │
│    (Para persistência dos dados tratados e consumo do BI)             │
└───────────────────────────────────────────────────────────────────────┘

```

---


In [544]:
###################
# IMPORTS
###################

import pandas as pd
import os
import re
import html

# Leitura dos CSV's baixados do site do Ministério da Saúde
# Iteração com `for` dentro do range de 2020 a 2026
# Armazenado em um dicionário chamado dfs

dfs = {ano: pd.read_csv(f"dados/bronze/{ano}.csv", sep=";", encoding="utf-8")
       for ano in range(2020, 2027)}


print('Bibliotecas e arquivos .csv importados.')

Bibliotecas e arquivos .csv importados.


---

```
┌───────────────────────────────────────────────────────────────────────┐
│ 1- EXTRACT - Verificação e conhecimento da base                       │
│    (Arquitetura Medalão - Camada Bronze)                              │
└───────────────────────────────────────────────────────────────────────┘
```

In [545]:
print('\n======= ANO 2020 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======\n')
print(dfs[2020].head(2).to_string())



======= ANO 2020 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======

   ano_compra                                     nome_instituicao     esfera    cnpj_instituicao municipio_instituicao  uf      compra    insercao  codigo_br                                                                                 descricao_catmat unidade_fornecimento generico        anvisa modalidade_compra     tipo_compra  capacidade unidade_medida unidade_fornecimento_capacidade     cnpj_fornecedor                  fornecedor     cnpj_fabricante                             fabricante  qtd_itens_comprados  preco_unitario  preco_total
0        2020               FUNDO  MUNICIPAL  DE  SAUDE DE  MARABA  MUNICIPAL  18.478.187/0001-07                MARABA  PA  01/01/2020  19/01/2024     270019                                 GLICONATO DE CÁLCIO, DOSAGEM:10%, APRESENTAÇÃO:SOLUÇÃO INJETÁVEL               AMPOLA        N  1.031100e+12            Pregão  ADMINISTRATIVA        10.0             ML                 AMPO

In [546]:
print('\n======= ANO 2021 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======\n')
print(dfs[2021].head(2).to_string())



======= ANO 2021 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======

   ano_compra                                          nome_instituicao     esfera    cnpj_instituicao municipio_instituicao  uf      compra    insercao  codigo_br                                                                                                                                                                                                                                                                                                       descricao_catmat unidade_fornecimento generico  anvisa modalidade_compra     tipo_compra  capacidade unidade_medida unidade_fornecimento_capacidade     cnpj_fornecedor                                                    fornecedor     cnpj_fabricante                fabricante  qtd_itens_comprados  preco_unitario  preco_total
0        2021                    FUNDO  MUNICIPAL  DE  SAUDE DE  MARABA  MUNICIPAL  18.478.187/0001-07                MARABA  PA  01/01/2021  29/12

In [547]:
print('\n======= ANO 2022 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======\n')
print(dfs[2022].head(2).to_string())



======= ANO 2022 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======

   ano_compra                        nome_instituicao     esfera    cnpj_instituicao municipio_instituicao  uf      compra    insercao  codigo_br                                                                                                             descricao_catmat unidade_fornecimento generico  anvisa modalidade_compra     tipo_compra  capacidade unidade_medida unidade_fornecimento_capacidade     cnpj_fornecedor                                        fornecedor     cnpj_fabricante                                         fabricante  qtd_itens_comprados  preco_unitario  preco_total
0        2022  FUNDO  MUNICIPAL  DE  SAUDE DE  MARABA  MUNICIPAL  18.478.187/0001-07                MARABA  PA  01/01/2022  23/01/2024     394088  BICARBONATO DE SÓDIO, CONCENTRAÇÃO:8,40%, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL, CARACTERÍSTICA ADICIONAL:EM SISTEMA FECHADO               FRASCO      NaN     NaN            Pregão  ADMINISTRA

In [548]:
print('\n======= ANO 2023 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======\n')
print(dfs[2023].head(2).to_string())



======= ANO 2023 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======

   ano_compra                        nome_instituicao     esfera    cnpj_instituicao municipio_instituicao  uf      compra    insercao  codigo_br                                                                          descricao_catmat unidade_fornecimento generico  anvisa modalidade_compra     tipo_compra  capacidade unidade_medida unidade_fornecimento_capacidade     cnpj_fornecedor                                                               fornecedor     cnpj_fabricante                                     fabricante  qtd_itens_comprados  preco_unitario  preco_total
0        2023  FUNDO  MUNICIPAL  DE  SAUDE DE  MARABA  MUNICIPAL  18.478.187/0001-07                MARABA  PA  01/01/2023  15/03/2024     423975            PIPETA, TIPO:PASTEUR, CAPACIDADE:3 ML, MATERIAL:PLÁSTICO, TIPO USO:DESCARTÁVEL              UNIDADE      NaN     NaN            Pregão  ADMINISTRATIVA         NaN            NaN                    

In [549]:
print('\n======= ANO 2024 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======\n')
print(dfs[2024].head(2).to_string())



======= ANO 2024 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======

   ano_compra          nome_instituicao     esfera    cnpj_instituicao municipio_instituicao  uf      compra    insercao  codigo_br                       descricao_catmat unidade_fornecimento generico        anvisa   modalidade_compra     tipo_compra  capacidade unidade_medida unidade_fornecimento_capacidade     cnpj_fornecedor                                       fornecedor     cnpj_fabricante                   fabricante  qtd_itens_comprados  preco_unitario  preco_total
0        2024  FUNDO MUNICIPAL DE SAUDE  MUNICIPAL  11.120.699/0001-40                MURICI  AL  01/01/2024  09/08/2024     268375  ACICLOVIR, DOSAGEM:50 MG/G, USO:CREME              BISNAGA        S  1.256801e+12              Pregão  ADMINISTRATIVA        10.0              G                 BISNAGA 10.00 G  00.236.193/0001-84  CIRURGICA RECIFE COMERCIO E REPRESENTACOES LTDA  73.856.593/0001-66  PRATI, DONADUZZI & CIA LTDA                  900     

In [550]:
print('\n======= ANO 2025 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======\n')
print(dfs[2025].head(2).to_string())



======= ANO 2025 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======

   ano_compra          nome_instituicao     esfera    cnpj_instituicao     municipio_instituicao  uf      compra    insercao  codigo_br                                                                                                  descricao_catmat unidade_fornecimento generico        anvisa modalidade_compra     tipo_compra  capacidade unidade_medida unidade_fornecimento_capacidade     cnpj_fornecedor                          fornecedor     cnpj_fabricante                          fabricante  qtd_itens_comprados  preco_unitario  preco_total
0        2025  FUNDO MUNICIPAL DE SAUDE  MUNICIPAL  11.328.684/0001-71  SAO FRANCISCO DO GUAPORE  RO  01/01/2025  01/09/2025     342134  HIDROCORTISONA, COMPOSIÇÃO:SAL SUCCINATO SÓDICO, CONCENTRAÇÃO:500 MG, FORMA FARMACÊUTICA:PÓ LIÓFILO P/ INJETÁVEL        FRASCO-AMPOLA        S  1.163701e+12            Pregão  ADMINISTRATIVA         NaN            NaN                   FRASCO-AM

In [551]:
print('\n======= ANO 2026 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======\n')
print(dfs[2026].head(2).to_string())



======= ANO 2026 - EXIBINDO AS PRIMEIRAS 2 LINHAS DA BASE. =======

   ano_compra             nome_instituicao     esfera    cnpj_instituicao municipio_instituicao  uf      compra    insercao  codigo_br                                                                                                                                                                                                                                       descricao_catmat unidade_fornecimento generico        anvisa modalidade_compra     tipo_compra  capacidade unidade_medida unidade_fornecimento_capacidade     cnpj_fornecedor                          fornecedor     cnpj_fabricante                        fabricante  qtd_itens_comprados  preco_unitario  preco_total
0        2026  MUNICIPIO DE PATOS DE MINAS  MUNICIPAL  18.602.011/0001-07        PATOS DE MINAS  MG  06/01/2026  27/01/2026     484594  TUBO ENDOBRONQUIAL, MODELO:DUPLO LÚMEN C/ GANCHO DE CARINA, TIPO USO:ESQUERDO, CALIBRE:35 FR, MATERIAL :LÁTEX NATURA

In [552]:
#############################################################
# Diagnóstico comparativo dos 7 DataFrames
#############################################################

print("======= 1. VOLUMETRIA (LINHAS E COLUNAS) =======")

# iteração com laço `for` no dicionário de dataframes 
# verificação de linhas e colunas através da propriedade `shape`
total_linhas = 0
for ano, df in dfs.items():
    print(f"Ano {ano}: {df.shape[0]:,} linhas | {df.shape[1]} colunas".replace(",", "."))
    total_linhas += df.shape[0]
print(f"\nTotal de linhas em toda base: {total_linhas}")    

print("\n======= 2. COMPARAÇÃO DAS COLUNAS =======")

# pega o conjunto de colunas de 2020 como referência
# `set` converte o Index com os nomes das colunas do Dataframe
# em um conjunto de strings únicas para futura comparação
cols_2020 = set(dfs[2020].columns)
diferencas_encontradas = False

# compara a referência 2020 com todos os outros anos
for ano in range(2021, 2027):
    cols_atual = set(dfs[ano].columns)
    if cols_2020 != cols_atual:
        diferencas_encontradas = True
        print(f"-> Divergência em {ano}:")

if not diferencas_encontradas:
    print("Sucesso: Todos os 7 arquivos possuem exatamente os mesmos nomes de colunas!")

# verificação dos tipos dos dados
print("\n======= 3. NOMES E TIPOS DE DADOS DE 2020 (REFERÊNCIA) =======")
display(dfs[2020].dtypes)



======= 1. VOLUMETRIA (LINHAS E COLUNAS) =======
Ano 2020: 84.819 linhas | 25 colunas
Ano 2021: 83.622 linhas | 25 colunas
Ano 2022: 88.991 linhas | 25 colunas
Ano 2023: 31.992 linhas | 25 colunas
Ano 2024: 26.258 linhas | 25 colunas
Ano 2025: 26.215 linhas | 25 colunas
Ano 2026: 819 linhas | 25 colunas

Total de linhas em toda base: 342716

======= 2. COMPARAÇÃO DAS COLUNAS =======
Sucesso: Todos os 7 arquivos possuem exatamente os mesmos nomes de colunas!

======= 3. NOMES E TIPOS DE DADOS DE 2020 (REFERÊNCIA) =======


ano_compra                           int64
nome_instituicao                       str
esfera                                 str
cnpj_instituicao                       str
municipio_instituicao                  str
uf                                     str
compra                                 str
insercao                               str
codigo_br                            int64
descricao_catmat                       str
unidade_fornecimento                   str
generico                               str
anvisa                             float64
modalidade_compra                      str
tipo_compra                            str
capacidade                         float64
unidade_medida                         str
unidade_fornecimento_capacidade        str
cnpj_fornecedor                        str
fornecedor                             str
cnpj_fabricante                        str
fabricante                             str
qtd_itens_comprados                  int64
preco_unita

---

```
┌───────────────────────────────────────────────────────────────────────┐
│ 2- TRANSFORM - Auditoria, Limpeza e Tratamento dos Dados              │
│   2.1 -> Arquitetura Medalhão (Camada Prata)                          │
│       2.1.1 -> Concatenação das bases (df_prata)                      │
│       2.1.2 -> Verificação de Nulos e Duplicados                      │
│       2.1.3 -> Verificação de Inconsistência de Domínio               │
│       2.1.4 -> Tratamento, Limpeza e Enriquecimento dos Dados         │
│       2.1.5 -> Otimização de Tipos (Downcasting)                      │
│   2.2 -> Arquitetura Medalhão (Camada Ouro)                           │
│       2.2.1 -> Seleção de Colunas e Engenharia de Features            │
│                (Tabela Única Denormalizada)                           │
│       2.2.2 -> Verificação da Qualidade dos Dados Quantitativos       │
│                (Criação da Flag 'flag_qualidade_dado')                │
│       2.2.3 -> Faixa de Dispersão de Preços dos Produtos              │
│                (Criação de 'coeficiente_variacao' - numérico          │
│                 e da 'faixa_dispersao_preco' - categórica)            │
│       2.2.4 -> Criação de Star Schema                                 │
│                (Tabela Fato e Dimensões)                              │
└───────────────────────────────────────────────────────────────────────┘
```


```
┌───────────────────────────────────────────────────────────────────────┐
│   2.1 -> Arquitetura Medalhão (Camada Prata)                          │
└───────────────────────────────────────────────────────────────────────┘
```
```
┌───────────────────────────────────────────────────────────────────────┐
│       2.1.1 -> Concatenação das bases (df_prata)                      │
└───────────────────────────────────────────────────────────────────────┘
```

In [553]:

# Concatenação das bases num único DataFrame
df_prata = pd.concat(dfs.values(), ignore_index=True)

print(
    "\n======= BASE CONSOLIDADA: ======="
    f"\n  {df_prata.shape[0]:,} linhas".replace(",", "."),
    f"\n  {df_prata.shape[1]} colunas"
)



======= BASE CONSOLIDADA: =======
  342.716 linhas 
  25 colunas



```
┌───────────────────────────────────────────────────────────────────────┐
│       2.1.2 -> Verificação de Nulos e Duplicados                      │
└───────────────────────────────────────────────────────────────────────┘
```

In [554]:
# Verificação de duplicatas perfeitas
# todas as colunas idênticas
print("\n======= DUPLICATAS IDÊNTICAS:", 
      df_prata.duplicated().sum(),
      "linhas ======="
)

# Mapeamento de nulos por coluna
# armazenado em um novo dataframe df_nulos
df_nulos = pd.DataFrame({

    # fazer a contagem de nulos em cada coluna
    'nulos': df_prata.isnull().sum(),

    # divide a contagem pelo total de linhas para gerar %
    'porcentagem': ((df_prata.isnull().sum() / len(df_prata)) * 100).round(2),

    # mostra o tipo de cada coluna para tratamento adequado
    'tipo': df_prata.dtypes
})

# mostrar o resultado com a função `display`
# filtra somente colunas com nulos (>0)
# ordena pela quantidade de nulos do maior para o menor
print("\n======= CONTAGEM E PERCENTUAL DE NULOS  =======") 
display(df_nulos[df_nulos['nulos'] > 0].sort_values(by='nulos', ascending=False))



======= DUPLICATAS IDÊNTICAS: 19 linhas =======

======= CONTAGEM E PERCENTUAL DE NULOS  =======


,nulos,porcentagem,tipo
unidade_medida,218035,63.62,str
capacidade,218035,63.62,float64
generico,172037,50.20,str
anvisa,172037,50.20,float64
insercao,2128,0.62,str
nome_instituicao,157,0.05,str
unidade_fornecimento,25,0.01,str
unidade_fornecimento_capacidade,25,0.01,str



```
┌───────────────────────────────────────────────────────────────────────┐
│       2.1.3 -> Verificação de Inconsistência de Domínio               │
└───────────────────────────────────────────────────────────────────────┘
```

In [555]:
#############################################################
# Auditoria de Domínio e Sanidade dos dados
#############################################################

# verificar se colunas de quantidade e financeiras são <= 0
print("\n======= 1. MÉTRICAS FINANCEIRAS E QUANTIDADES =======")
print("Qtd itens comprados <= 0:  ", (df_prata['qtd_itens_comprados'] <= 0).sum())
print("Preço unitário <= 0:       ", (df_prata['preco_unitario'] <= 0).sum())
print("Preço total <= 0:          ", (df_prata['preco_total'] <= 0).sum())

# Checagem da regra de negócio: 
# Preço Total = Qtd * Preço Unitário 
# com tolerância de R$ 0,05 para possíveis arredondamentos
dif_preco = (df_prata['preco_total'] - (df_prata['qtd_itens_comprados'] * df_prata['preco_unitario'])).abs()
print("Incoerências de cálculo (Preço Total != Qtd * Preço Unit):", (dif_preco > 0.05).sum())

# verificar as UFs cadastradas
print("\n======= 2. AUDITORIA GEOGRÁFICA (UFs) =======")

# armazenar as ufs ordenadas alfabeticamente em uma lista
# converte cada valor único iterado no laço `for` em uma string `str`
ufs_unicas = sorted([str(uf) for uf in df_prata['uf'].dropna().unique()])
print(f"Total de UFs encontradas ({len(ufs_unicas)})")
print("UFs encontradas:", ufs_unicas)

# verificar se a de data de inserção e anterior à data de compra
print("\n======= 3. AUDITORIA DE DATAS =======")
compra_dt = pd.to_datetime(df_prata['compra'], format='%d/%m/%Y', errors='coerce')
insercao_dt = pd.to_datetime(df_prata['insercao'], format='%d/%m/%Y', errors='coerce')

print("Datas de compra inválidas (NaT):", compra_dt.isna().sum())
print("Datas de inserção inválidas (NaT):", insercao_dt.isna().sum())
print("Datas de inserção anteriores à compra:", (insercao_dt < compra_dt).sum())

print("\n* Lembrando que na análise anterior tivemos 2128 linhas com 'insercao' nulas (= datas inválidas) ")




======= 1. MÉTRICAS FINANCEIRAS E QUANTIDADES =======
Qtd itens comprados <= 0:   0
Preço unitário <= 0:        0
Preço total <= 0:           0
Incoerências de cálculo (Preço Total != Qtd * Preço Unit): 0

======= 2. AUDITORIA GEOGRÁFICA (UFs) =======
Total de UFs encontradas (24)
UFs encontradas: ['AC', 'AL', 'BA', 'CE', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']

======= 3. AUDITORIA DE DATAS =======
Datas de compra inválidas (NaT): 0
Datas de inserção inválidas (NaT): 2128
Datas de inserção anteriores à compra: 12

* Lembrando que na análise anterior tivemos 2128 linhas com 'insercao' nulas (= datas inválidas) 


In [556]:

# verificação de colunas categóricas

print("\n======= 4. CARDINALIDADE E VALORES ÚNICOS EM CATEGÓRICAS =======")
for col in ['esfera', 'generico', 'modalidade_compra', 'tipo_compra']:
    print(f"\n--- Frequência: {col} ---")
    print(df_prata[col].value_counts(dropna=False))


======= 4. CARDINALIDADE E VALORES ÚNICOS EM CATEGÓRICAS =======

--- Frequência: esfera ---
esfera
MUNICIPAL    318369
ESTADUAL      24182
FEDERAL         122
0                43
Name: count, dtype: int64

--- Frequência: generico ---
generico
NaN    172037
S       94318
N       76361
Name: count, dtype: int64

--- Frequência: modalidade_compra ---
modalidade_compra
Pregão                          315343
Dispensa de Licitação            13177
Registro de Preços               10951
Tomada de Preços                  1784
Concorrência                       722
Inexigibilidade de Licitação       416
Concurso                           196
Convite                             70
Leilão                              57
Name: count, dtype: int64

--- Frequência: tipo_compra ---
tipo_compra
ADMINISTRATIVA    337498
JUDICIAL            5218
Name: count, dtype: int64


In [557]:

# verificação de strings mal formatadas com códigos utilizando regex

print("\n======= 5. COLUNA E QUANTIDADE DE LINHAS MAL FORMATADAS =======")

def verificar_strings():
    # compila o regex uma única vez (eficiência)
    # '&#\d+;' — entidade numérica (&#193;, &#205;, etc.)
    # '</?[a-zA-Z][^>]*>' — tag HTML 
    regex_cod = re.compile(r'&#\d+;|</?[a-zA-Z][^>]*>')

    #Dicionário vazio que vai armazenar: {nome_da_coluna: quantidade_de_células}.
    colunas_com_cod = {}

    #Itera sobre todas as colunas do DataFrame cujo tipo é str (texto).
    # .dropna() — remove células vazias (evita erro no .str.contains)
    # .str.contains(regex_cod, regex=True) — retorna uma Série de True/False: True se a célula contém códigos, False se não
    # .sum() — soma os True (que valem 1) → dá o total de células com códigos naquela coluna

    for col in df_prata.select_dtypes(include='str').columns:
        count = df_prata[col].dropna().str.contains(regex_cod, regex=True).sum()
        if count > 0:
            colunas_com_cod[col] = count

    for col, n in colunas_com_cod.items():
        print(f"\n{col}: {n} de {df_prata[col].notna().sum()} células com código ou HTML")

verificar_strings()


======= 5. COLUNA E QUANTIDADE DE LINHAS MAL FORMATADAS =======

descricao_catmat: 854 de 342716 células com código ou HTML



```
┌───────────────────────────────────────────────────────────────────────┐
│       2.1.4 -> Tratamento, Limpeza e Enriquecimento dos Dados         │
└───────────────────────────────────────────────────────────────────────┘
```

In [558]:
##################################
# CAMADA PRATA: TRANSFORM
##################################

##################################
# 1. Remoção de duplicatas exatas
##################################

df_prata = df_prata.drop_duplicates().reset_index(drop=True)

##################################
# 2. Tratamento e conversão de datas 
# (formato da base original DD/MM/YYYY)
##################################

df_prata['compra'] = pd.to_datetime(df_prata['compra'], format='%d/%m/%Y', errors='coerce')

# foi utilizado um 'fallback' para as linhas em que 'insercao' é nulo
# e preenchido com a data de 'compra'. Após convertido o campo com pd.to_datetime
df_prata['insercao'] = df_prata['insercao'].fillna(df_prata['compra'])
df_prata['insercao'] = pd.to_datetime(df_prata['insercao'], format='%d/%m/%Y', errors='coerce')

# correção dos 12 registros encontrados na verificação anterior
# onde a data de inserção era anterior à data de compra (regra de negócio)
# por ser poucos registros, foi atribuida a mesma data da compra
mascara_data_invalida = df_prata['insercao'] < df_prata['compra']
df_prata.loc[mascara_data_invalida, 'insercao'] = df_prata.loc[mascara_data_invalida, 'compra']

##################################
# 3. Enriquecimento dos Dados 
# (com dados históricos via mapeamento)
##################################

# Nome da Instituição buscando por CNPJ
# construir um dicionário mantendo somente as linhas onde o nome já existe (dropna)
# se o mesmo cnpj aparecer várias vezes, manter somente um (drop_duplicates)
# criar uma série indexada pelo cnpj e o nome_instituicao como valor (set_index)
# converter para um dicionário (to_dict)
mapa_instituicoes = (
    df_prata.dropna(subset=['nome_instituicao'])
    .drop_duplicates(subset=['cnpj_instituicao'])
    .set_index('cnpj_instituicao')['nome_instituicao']
    .to_dict()
)
# preencher o nome onde estiver nulo buscando pelo cnpj do dicionário criado
df_prata['nome_instituicao'] = df_prata['nome_instituicao'].fillna(df_prata['cnpj_instituicao'].map(mapa_instituicoes))

# ANVISA e Genérico por Código BR (CATMAT)
# mesma lógica do nome da instituição, agora para códico Anvisa por codigo_br (CATMAT) 
mapa_anvisa = (
    df_prata.dropna(subset=['anvisa'])
    .drop_duplicates(subset=['codigo_br'])
    .set_index('codigo_br')['anvisa']
    .to_dict()
)
# mesma lógica do nome da instituição, agora para genérico por codigo_br (CATMAT) 
mapa_generico = (
    df_prata.dropna(subset=['generico'])
    .drop_duplicates(subset=['codigo_br'])
    .set_index('codigo_br')['generico']
    .to_dict()
)

# preencher anvisa e genérico buscando pelos dicionários criados
df_prata['anvisa'] = df_prata['anvisa'].fillna(df_prata['codigo_br'].map(mapa_anvisa))
df_prata['generico'] = df_prata['generico'].fillna(df_prata['codigo_br'].map(mapa_generico))

##################################
# 4. Preenchimento de Nulos Residuais com "NÃO INF." para str e "0" para int
# para os preenchimentos anteriores, caso não seja encontrado referência
# e também para os campos 'unidade_medida', 'unidade_fornecimento', 'unidade_fornecimento_capacidade'
##################################

cols_texto_nao_inf = [
    'nome_instituicao', 'generico', 'unidade_medida', 
    'unidade_fornecimento', 'unidade_fornecimento_capacidade'
]
for col in cols_texto_nao_inf:
    df_prata[col] = df_prata[col].fillna('NÃO INF.')

df_prata['anvisa'] = df_prata['anvisa'].fillna(0).astype('int64')

##################################
# 5. Preenchimento coluna 'esfera' 
# com "NÃO INF." onde foi encontrado "0"
##################################

# encontrado 43 registros
df_prata['esfera'] = df_prata['esfera'].replace('0', 'NÃO INF.')   

##################################
# 6. Validação final dos nulos
##################################

# Deve retornar apenas 'capacidade' como esperado
nulos_restantes = df_prata.isnull().sum()
print("\n======= NULOS RESTANTES NA CAMADA PRATA =======")
print(nulos_restantes[nulos_restantes > 0])




======= NULOS RESTANTES NA CAMADA PRATA =======
capacidade    218027
dtype: int64


In [559]:

##################################
# 7. Sanitização de Strings 
##################################

# função para limpar os códigos de textos
def limpar_cods(texto):
    # se o valor for NaN, None, número, etc, retorna com está
    if not isinstance(texto, str):
        return texto
    # Regex para remover tags HTML (substitui por espaço)
    texto = re.sub(r'</?[a-zA-Z][^>]*>', ' ', texto)
    # Decodifica entidades (&#193; → Á, etc.)
    texto = html.unescape(texto)
    # NBSP → espaço normal
    texto = texto.replace('\xa0', ' ')
    # Colapsa espaços múltiplos
    texto = re.sub(r'\s{2,}', ' ', texto)
    # remove espaço no início e no fim
    texto = texto.strip()   
    # aplica caixa alta (padrão dos dados)
    return texto.upper()

# iterar sobre a coluna 'descricao_catmat' onde foram 
# encontrados códigos, aplicando a transformação (limpeza)
df_prata['descricao_catmat'] = df_prata['descricao_catmat'].apply(limpar_cods)

# verificar strings mal formatadas residuais
verificar_strings()

print("\n======= Transformações da Camada Prata concluídas com sucesso! =======")



======= Transformações da Camada Prata concluídas com sucesso! =======



```
┌───────────────────────────────────────────────────────────────────────┐
│       2.1.5 -> Otimização de Tipos (Downcasting)                      │
└───────────────────────────────────────────────────────────────────────┘
```

In [560]:
# Verificação dos tipos atuais na camada Prata

print("\n======= NOMES E TIPOS DE DADOS CAMADA PRATA =======")
display(df_prata.dtypes)



======= NOMES E TIPOS DE DADOS CAMADA PRATA =======


ano_compra                                  int64
nome_instituicao                              str
esfera                                        str
cnpj_instituicao                              str
municipio_instituicao                         str
uf                                            str
compra                             datetime64[us]
insercao                           datetime64[us]
codigo_br                                   int64
descricao_catmat                              str
unidade_fornecimento                          str
generico                                      str
anvisa                                      int64
modalidade_compra                             str
tipo_compra                                   str
capacidade                                float64
unidade_medida                                str
unidade_fornecimento_capacidade               str
cnpj_fornecedor                               str
fornecedor                                    str


In [561]:

# Verificação do número de valores distintos em cada coluna
# para análise do que pode ser transformado em coluna categórica (category)

print("\n======= NOME E VALORES DISTINTOS CAMADA PRATA =======\n")
print(df_prata.nunique(dropna=False).to_string())   



======= NOME E VALORES DISTINTOS CAMADA PRATA =======

ano_compra                             7
nome_instituicao                     663
esfera                                 4
cnpj_instituicao                     831
municipio_instituicao                729
uf                                    24
compra                              1887
insercao                            1698
codigo_br                          12994
descricao_catmat                   12993
unidade_fornecimento                  45
generico                               3
anvisa                             13500
modalidade_compra                      9
tipo_compra                            2
capacidade                           130
unidade_medida                        16
unidade_fornecimento_capacidade      389
cnpj_fornecedor                     3502
fornecedor                          3294
cnpj_fabricante                     2290
fabricante                          2124
qtd_itens_comprados                21667
p

In [562]:

# Verificação dos valores máximos inteiros 'INT'
# para otimização (downcasting)

print(f"Máximo da coluna 'ano_compra':           {df_prata['ano_compra'].max()}")
print(f"Máximo da coluna 'codigo_br':            {df_prata['codigo_br'].max()}")
print(f"Máximo da coluna 'qtd_itens_comprados':  {df_prata['qtd_itens_comprados'].max()}")
print(f"Máximo da coluna 'anvisa':               {df_prata['anvisa'].max()}")


Máximo da coluna 'ano_compra':           2026
Máximo da coluna 'codigo_br':            635152
Máximo da coluna 'qtd_itens_comprados':  4356140000
Máximo da coluna 'anvisa':               1986000200018


In [563]:
####################################################################
# OTIMIZAÇÃO DE TIPOS E DOWNCASTING DE MEMÓRIA (CAMADA PRATA)
####################################################################

# Medição de memória inicial (para comparação após otimização)
# memory_usage(deep=True): Calcula a memória exata
# .sum(): Soma o consumo de todas as colunas e do índice.
# / 1024**2: Converte o total de bytes (unidade nativa do pandas)
#  para megabytes (base-2, onde 1 MB = 1024 KB).
memoria_inicial = df_prata.memory_usage(deep=True).sum() / (1024 ** 2)

# 1. Inteiros Otimizados (Downcasting)
df_prata['ano_compra'] = df_prata['ano_compra'].astype('int16')
df_prata['codigo_br'] = df_prata['codigo_br'].astype('int32')
df_prata['qtd_itens_comprados'] = df_prata['qtd_itens_comprados'].astype('int64')
df_prata['anvisa'] = df_prata['anvisa'].astype('int64')

# 2. Conversão das colunas de categoria (Baixa/Média Cardinalidade)
categorias = [
    'esfera', 'uf', 'generico', 'modalidade_compra', 'tipo_compra', 
    'unidade_medida', 'unidade_fornecimento', 
    'unidade_fornecimento_capacidade', 'municipio_instituicao'
]

for col in categorias:
    df_prata[col] = df_prata[col].astype('category')


# Medição de memória final
memoria_final = df_prata.memory_usage(deep=True).sum() / (1024 ** 2)
reducao_percentual = ((memoria_inicial - memoria_final) / memoria_inicial) * 100

print(f"Memória Inicial:     {memoria_inicial:.2f} MB")
print(f"Memória Final:       {memoria_final:.2f} MB")
print(f"Redução de Memória:  {reducao_percentual:.1f}%")

Memória Inicial:     171.65 MB
Memória Final:       124.71 MB
Redução de Memória:  27.3%


---

```
┌───────────────────────────────────────────────────────────────────────┐
│   2.2 -> Arquitetura Medalhão (Camada Ouro)                           │
└───────────────────────────────────────────────────────────────────────┘
```


```
┌───────────────────────────────────────────────────────────────────────┐
│       2.2.1 -> Seleção de Colunas e Engenharia de Features            │
│                (Tabela Única Denormalizada)                           │
└───────────────────────────────────────────────────────────────────────┘
```

In [564]:

# Verificação de redundancia entre a coluna 'ano_compra' e 'compra'
# conferir se todas os registros "batem" em relação a data da compra e o ano declarado
# caso não haja divergências, 'ano_compra' pode ser descartado

# Extrair o ano diretamente da coluna 'compra' (datetime)
ano_extraido = df_prata['compra'].dt.year

# Verificar registros onde 'ano_compra' é diferente do ano real da compra
divergencias = (df_prata['ano_compra'] != ano_extraido).sum()

print(f"\nTotal de registros analisados: {len(df_prata):,}")
print(f"Registros com divergência de ano: {divergencias}")

# Confirmação dos nomes das colunas para seleção das colunas mantidas
print("\n======= CONFIRMAÇÃO NOMES DE COLUNAS PARA SELEÇÃO =======\n")
df_prata.columns



Total de registros analisados: 342,697
Registros com divergência de ano: 0

======= CONFIRMAÇÃO NOMES DE COLUNAS PARA SELEÇÃO =======



Index(['ano_compra', 'nome_instituicao', 'esfera', 'cnpj_instituicao',
       'municipio_instituicao', 'uf', 'compra', 'insercao', 'codigo_br',
       'descricao_catmat', 'unidade_fornecimento', 'generico', 'anvisa',
       'modalidade_compra', 'tipo_compra', 'capacidade', 'unidade_medida',
       'unidade_fornecimento_capacidade', 'cnpj_fornecedor', 'fornecedor',
       'cnpj_fabricante', 'fabricante', 'qtd_itens_comprados',
       'preco_unitario', 'preco_total'],
      dtype='str')

In [565]:
#######################################################################
# CAMADA OURO - ABORDAGEM 1: TABELA ÚNICA DENORMALIZADA (df_ouro)
#######################################################################

# 1. Cópia base a partir da Camada Prata
df_ouro = df_prata.copy()

# 2. Seleção de colunas mantidas (21 colunas)
# Descarte de insercao, capacidade e unidade_medida e ano_compra
cols_mantidas = [
    'nome_instituicao', 'esfera', 'cnpj_instituicao',
    'municipio_instituicao', 'uf', 'compra', 'codigo_br',
    'descricao_catmat', 'unidade_fornecimento', 'generico', 'anvisa',
    'modalidade_compra', 'tipo_compra', 'unidade_fornecimento_capacidade',
    'cnpj_fornecedor', 'fornecedor', 'cnpj_fabricante', 'fabricante',
    'qtd_itens_comprados', 'preco_unitario', 'preco_total'
]
df_ouro = df_ouro[cols_mantidas]

# 3. Engenharia de Features Temporais (Extração numérica)
df_ouro['nr_ano'] = df_ouro['compra'].dt.year.astype('int16')
df_ouro['nr_mes'] = df_ouro['compra'].dt.month.astype('int8')
df_ouro['nr_trimestre'] = df_ouro['compra'].dt.quarter.astype('int8')
# Dia da semana numérico -> 0=Segunda, 6=Domingo
df_ouro['nr_dia_semana'] = df_ouro['compra'].dt.dayofweek.astype('int8')

# 4. Engenharia de Feature (para regra negócio -> criar 'categoria_insumo')
# Regra: Se generico != 'NÃO INF.' ou anvisa > 0 -> 'MEDICAMENTO',
# caso contrário 'CORRELATO'
condicao_medicamento = (df_ouro['generico'] != 'NÃO INF.') | (df_ouro['anvisa'] > 0)

# criação da coluna atribuindo 'CORRELATO' para todas as linhas
df_ouro['categoria_insumo'] = 'CORRELATO'

df_ouro.loc[condicao_medicamento, 'categoria_insumo'] = 'MEDICAMENTO'
df_ouro['categoria_insumo'] = df_ouro['categoria_insumo'].astype('category')

print(f"\n======= Camada Ouro (Tabela Única - primeira versão) gerada com sucesso! =======")
print(f"Shape do df_ouro: {df_ouro.shape[0]} linhas x {df_ouro.shape[1]} colunas.")



======= Camada Ouro (Tabela Única - primeira versão) gerada com sucesso! =======
Shape do df_ouro: 342697 linhas x 26 colunas.



```
┌───────────────────────────────────────────────────────────────────────┐
│       2.2.2 -> Verificação da Qualidade dos Dados Quantitativos       │
│                (Criação da Flag 'flag_qualidade_dado')                │
└───────────────────────────────────────────────────────────────────────┘
```

In [566]:


##########################################################################
# Verificar a distância do 'preço_unitário' da transação em relação a mediana do preço
# Objetivo: Detectar a variação normal de mercado para possívies "outliers"
# (Variações entre mediana e máximo ou mínimo extremos = possíveis erros de digitação)
# um produto, codigo_br 429724 por exemplo, estava com 
# preço mínimo registrado de R$ 0,0001 e máximo de R$ 4.804.290,00
##########################################################################

# Mediana por grupo (produto + apresentação)
mediana_grupo = df_ouro.groupby(['codigo_br', 'unidade_fornecimento_capacidade'])['preco_unitario'].median().reset_index()
mediana_grupo = mediana_grupo.rename(columns={'preco_unitario': 'preco_mediano_grupo'})

# Merge temporário em novo DataFrame só para análise
df_analise = df_ouro.merge(mediana_grupo, on=['codigo_br', 'unidade_fornecimento_capacidade'], how='left')

# Razão preco_unitario/mediana (nível de transação)
df_analise['razao_transacao_mediana'] = df_analise['preco_unitario'] / df_analise['preco_mediano_grupo'].replace(0, pd.NA)

# percentis específicos na cauda
quantis = [0.9, 0.95, 0.97, 0.99, 0.995, 0.999, 0.9999]

##############################################################
# Distribuição - cauda superior (preço muito acima da mediana)
##############################################################

print(f"\n======= Cauda superior (preço acima da mediana): =======\n")

cauda_superior = df_analise['razao_transacao_mediana'].quantile(quantis)

#iterar sobre todos os quantis
for i in range(len(quantis)):
    if i == 0:
        print(f"Q {quantis[i]:.4f}: {cauda_superior.iloc[i]:.2f}")
    else:
        # calcular a taxa de crescimento 'Q atual' / 'Q anterior'
        tx = cauda_superior.iloc[i] / cauda_superior.iloc[i-1]
        print(f"Q {quantis[i]:.4f}: {cauda_superior.iloc[i]:.2f} | Tx Crescimento: {tx:.2f}x")   

################################################################################
# Distribuição - cauda inferior (preço muito abaixo da mediana) - usar o inverso
################################################################################

df_analise['razao_inversa'] = 1 / df_analise['razao_transacao_mediana']
print(f"\n======= Cauda inferior (preço abaixo da mediana, em razão inversa): =======\n")

cauda_inferior = df_analise['razao_inversa'].quantile(quantis)

#iterar sobre todos os quantis
for i in range(len(quantis)):
    if i == 0:
        print(f"Q {quantis[i]:.4f}: {cauda_inferior.iloc[i]:.2f}")
    else:
        # calcular a taxa de crescimento 'Q atual' / 'Q anterior'
        tx = cauda_inferior.iloc[i] / cauda_inferior.iloc[i-1]
        print(f"Q {quantis[i]:.4f}: {cauda_inferior.iloc[i]:.2f} | Tx Crescimento: {tx:.2f}x")




======= Cauda superior (preço acima da mediana): =======

Q 0.9000: 2.00
Q 0.9500: 2.97 | Tx Crescimento: 1.49x
Q 0.9700: 4.36 | Tx Crescimento: 1.47x
Q 0.9900: 17.77 | Tx Crescimento: 4.07x
Q 0.9950: 43.22 | Tx Crescimento: 2.43x
Q 0.9990: 252.07 | Tx Crescimento: 5.83x
Q 0.9999: 1530.59 | Tx Crescimento: 6.07x

======= Cauda inferior (preço abaixo da mediana, em razão inversa): =======

Q 0.9000: 1.56
Q 0.9500: 2.02 | Tx Crescimento: 1.30x
Q 0.9700: 2.95 | Tx Crescimento: 1.46x
Q 0.9900: 28.36 | Tx Crescimento: 9.61x
Q 0.9950: 84.58 | Tx Crescimento: 2.98x
Q 0.9990: 420.48 | Tx Crescimento: 4.97x
Q 0.9999: 297115.04 | Tx Crescimento: 706.61x



========== RAZÃO DA VARIAÇÃO ENTRE PREÇO UNITÁRIO E MEDIANA ===========

CONCLUSÃO E INSIGHTS GERADOS

Foi criada a medida de razão (`preço_unitario / mediana`) e razão inversa (`1 / razão`)<br/>
para calcular a distância do preço unitário de cada transação em relação a mediana do produto<br/>
tanto na cauda superior, como na cauda inferior.

Após foi selecionados os percentis acima de 0.9 (90%) para visualizar a ponta das caudas.<br/>

Analisando a taxa de crescimento marginal em cada cauda, elas não são simétricas, reforça que usar<br/>
limiares (pontos de corte) diferentes para cima e para baixo é o correto.

**Cauda superior (preço acima da mediana):**<br/>
Crescimento consistente até p97 (1.47%), acelera moderadamente em p97→p99 (+4,07x).<br/>
O salto mais acentuado só ocorre depois de p99 (`p99,5→p99,9: 5,83x | p99,9→p99,99: 6,07x`).<br/>
Corte recomendado: p99 → razão 17,8x. (representando ~1% das transações)

**Cauda inferior (preço abaixo da mediana, razão inversa):**<br/>
Aqui o salto é mais abrupto e ocorre mais cedo: p97→p99 salta 9,61x <br/>
(bem mais acentuado que na cauda superior, no mesmo intervalo). <br/>
Corte recomendado: p99 → razão inversa 28,4x. (representando ~1% das transações)

***Limiar recomendado para classificação:***
- Suspeito se preco_unitario > mediana_do_grupo × 17,8 (preço muito acima do normal)
- Suspeito se preco_unitario < mediana_do_grupo / 28,4 (preço muito abaixo do normal)

====================================================================


In [567]:


##########################################################################
# Verificar 'preco_unitario' com valores muito baixos fora da realidade
# na inspeção do banco foram encontrado vários itens com preço de R$ 0,0001
# isso pode indicar possivelmente um "placeholder"
##########################################################################

# Ver a distribuição de preços na cauda inferior
print(f"\n======= Exibindo percentil ('.quantile()') VS Preço Unitário =======")
print(f"======= Para distribuição de preços na cauda inferior =======\n")

print(df_ouro[['preco_unitario']].quantile([0.001, 0.005, 0.01, 0.02, 0.05]))

# Contagem em diferentes limiares candidatos
print(f"\n======= Contagem de preços em diferentes limiares (valores de corte) =======\n")

limiares = [0.001, 0.005, 0.01, 0.03, 0.05, 0.1]

for i, limiar in enumerate(limiares):
    qtd = (df_ouro['preco_unitario'] <= limiar).sum()
    if i == 0:
        print(f"Registros com preco_unitario <= {limiar}: {qtd} ({qtd/len(df_ouro)*100:.2f}%)")
    else:
        qtd_anterior = (df_ouro['preco_unitario'] <= limiares[i-1]).sum()
        print(f"Registros com preco_unitario <= {limiar}: {qtd} ({qtd/len(df_ouro)*100:.2f}%) - Tx Crescimento: {(qtd/qtd_anterior):.2f}x")   



======= Exibindo percentil ('.quantile()') VS Preço Unitário =======
======= Para distribuição de preços na cauda inferior =======

       preco_unitario
0.001           0.001
0.005           0.020
0.010           0.030
0.020           0.040
0.050           0.069

======= Contagem de preços em diferentes limiares (valores de corte) =======

Registros com preco_unitario <= 0.001: 360 (0.11%)
Registros com preco_unitario <= 0.005: 886 (0.26%) - Tx Crescimento: 2.46x
Registros com preco_unitario <= 0.01: 1154 (0.34%) - Tx Crescimento: 1.30x
Registros com preco_unitario <= 0.03: 4478 (1.31%) - Tx Crescimento: 3.88x
Registros com preco_unitario <= 0.05: 11995 (3.50%) - Tx Crescimento: 2.68x
Registros com preco_unitario <= 0.1: 32147 (9.38%) - Tx Crescimento: 2.68x



============= DISTRIBUIÇÃO DE PREÇOS NA CAUDA INFERIOR =============

CONCLUSÃO E INSIGHTS GERADOS

O limiar de R$0,01 apresentou a menor taxa de crescimento incremental de registros na distribuição de preços.<br/>
Entre R$0,005 e R$0,01, a contagem cresce apenas 1,30x (a menor variação de toda a série analisada).<br/>
Esse platô indica o fim de um agrupamento de valores atípicos <br/>
(podendo ser associados a erros de digitação e valores "sentinela / placeholder", como os casos 0,0001 e 0,0125<br/>
identificados na auditoria manual) e o início da distribuição normal de preços baixos, porém válidos.

***Sustenta a escolha de preços unitários <= 0,01 serem classificados como suspeitos.***

====================================================================

In [568]:


##########################################################################
# Verificar 'qtd_itens_comprados' com valores absurdos (muito altos)
# na inspeção do banco foram encontrado registros com quantidades 
# por exemplo de 4.356.140.000 - 819.600.000 - 624.240.000
##########################################################################

# Ver a distribuição de qtde_itens_comprados na cauda superior
print(f"\n======= Exibindo percentil ('.quantile()') VS Qtde Itens Comprados =======")
print(f"======= Para Quantidade de Itens na cauda superior =======\n")

print(df_ouro[['qtd_itens_comprados']].quantile([0.9, 0.95, 0.97, 0.99, 0.995, 0.999, 0.9999]).map('{:.0f}'.format))   




======= Exibindo percentil ('.quantile()') VS Qtde Itens Comprados =======
======= Para Quantidade de Itens na cauda superior =======

       qtd_itens_comprados
0.9000               72000
0.9500              220000
0.9700              460000
0.9900             1800000
0.9950             4152761
0.9990            24691949
0.9999           100000000



=================== QUANTIDADE DE ITENS EXTREMOS ===================

CONCLUSÃO E INSIGHTS GERADOS

Crescimento mais uniforme, sem ruptura tão nítida quanto a razão.<br/>
Taxa de crescimento:

| p90→p95: 3,05x  |<br/> 
| p95→p97: 2,09x  |<br/>
| p97→p99: 3,91x  |<br/> 
| p99→p99.5: 2,3x    |<br/>
| p99.5→p99.9: 5,95x  |<br/>
| p99.9→p99.99: 4,05x  |<br/>

Maior salto relativo ocorre em p99,5→p99,9 (5,95x). 

***Sustenta o corte em p99,9 (≈24,7M) para classificação como suspeito.***

====================================================================


In [569]:

##########################################################################
# CRIAÇÃO DA FLAG DE QUALIDADE DO DADO
# coluna 'flag_qualidade_dado' categórica com múltiplos motivos concatenados
# (um registro pode violar mais de 1 critério simultaneamente)
##########################################################################

###############################################
# 1. Mediana por grupo (produto + apresentação)

stats_mediana = df_ouro.groupby(['codigo_br', 'unidade_fornecimento_capacidade'])['preco_unitario'].median().reset_index()
stats_mediana = stats_mediana.rename(columns={'preco_unitario': 'preco_mediano_grupo'})

# Merge no df_ouro
df_ouro = df_ouro.merge(stats_mediana, on=['codigo_br', 'unidade_fornecimento_capacidade'], how='left')

##########################################################################
# 2. Avaliação de cada critério nível de transação (booleanos individuais)
crit_pico = df_ouro['preco_unitario'] > (df_ouro['preco_mediano_grupo'] * 17.8)
crit_piso_relativo = df_ouro['preco_unitario'] < (df_ouro['preco_mediano_grupo'] / 28.4)
crit_piso_absoluto = df_ouro['preco_unitario'] <= 0.01
crit_qtd = df_ouro['qtd_itens_comprados'] > 24_700_000

###################################
# 3. Construção da flag concatenada
# função para montar string de motivos
def montar_flag(pico, piso_rel, piso_abs, qtd):
    motivos = []
    if pico:
        motivos.append('Preço Acima do Padrão')
    if piso_rel or piso_abs:
        motivos.append('Preço Abaixo do Padrão')
    if qtd:
        motivos.append('Quantidade Extrema')
    return 'Válido' if not motivos else 'Suspeito - ' + ' + '.join(motivos)

# chama função de montar flag combinando múltiplos iteráveis com o '.zip()'
# itera em cada linha, trazendo os boleanos de avaliaçao e atribuindo na função
# para retorno da flat

df_ouro['flag_qualidade_dado'] = [
    montar_flag(p, pr, pa, q) 
    for p, pr, pa, q in zip(crit_pico, crit_piso_relativo, crit_piso_absoluto, crit_qtd)
]

# define o tipo do dado como categoria
df_ouro['flag_qualidade_dado'] = df_ouro['flag_qualidade_dado'].astype('category')

# Remover coluna auxiliar
df_ouro = df_ouro.drop(columns=['preco_mediano_grupo'])

################
# 4. Conferência
print(f"\n======= Verificação da 'flag_qualidade_dado' com '.value_counts()' =======\n")
print(df_ouro['flag_qualidade_dado'].value_counts())




======= Verificação da 'flag_qualidade_dado' com '.value_counts()' =======

flag_qualidade_dado
Válido                                                    335487
Suspeito - Preço Abaixo do Padrão                           3448
Suspeito - Preço Acima do Padrão                            3419
Suspeito - Quantidade Extrema                                286
Suspeito - Preço Abaixo do Padrão + Quantidade Extrema        57
Name: count, dtype: int64


<br/>


```
┌───────────────────────────────────────────────────────────────────────┐
│       2.2.3 -> Faixa de Dispersão de Preços dos Produtos              │
│                (Criação de 'coeficiente_variacao' - numérico          │
│                 e da 'faixa_dispersao_preco' - categórica)            │
└───────────────────────────────────────────────────────────────────────┘
```

Coeficiente de Variação é a razão entre o desvio padrão e a média de um conjunto de valores: `CV = Desvio Padrão / Média`

Diferente do desvio padrão puro, o CV é *adimensional e relativo*, permite comparar a <br/> 
dispersão de preços entre produtos de faixas de valor totalmente diferentes <br/>
(ex: um produto que custa R$0,50 vs. outro que custa R$5.000) de forma justa, já que normaliza pela escala do próprio produto.

*Importância para essa análise:*

**Ranqueamento de risco por produto:** em vez de olhar caso a caso (nível transação, granular fino), <br/>
o CV permite classificar todos os ~13 mil produtos de uma vez em Baixa/Média/Alta dispersão, <br/>
vira o filtro de "Faixa de Dispersão" a Página de análise de produtos do BI.

**Diferente da flag de qualidade do dado:** a flag marca erros/outliers suspeitos (dado provavelmente errado). <br/>
O CV mede variação de mercado legítima, produtos onde o preço varia muito entre fornecedores/instituições/período,<br/>
mesmo sem nenhum valor ser "errado". É achado de negócio (oportunidade de negociação, possível sobrepreço regional),<br/>
não problema de qualidade de dado.

**Complementaridade:** idealmente, o CV deve ser calculado excluindo os registros já flagados como suspeitos<br/>
(senão os outliers de erro contaminam a métrica de variação legítima) e <br/>
por isso faz sentido calcular o CV depois da flag, não antes.



In [570]:

#######################################################################################
# EXPLORAÇÃO: Coeficiente de Variação por produto + apresentação
# (somente com 'flag_qualidade_dado = Válido', excluindo registros marcados como suspeitos)
#######################################################################################

# Dataframe temporário excluindo registros suspeitos
df_validos = df_ouro[df_ouro['flag_qualidade_dado'] == 'Válido']

# agrupando por produto e adicionando medidas estatísticas
cv_produto = df_validos.groupby(['codigo_br', 'unidade_fornecimento_capacidade'])['preco_unitario'].agg(
    preco_medio='mean',
    preco_desvio_padrao=lambda x: x.std(ddof=0),
    qtd_transacoes='count'
).reset_index()

# cálculo do CV
cv_produto['coeficiente_variacao'] = (
    cv_produto['preco_desvio_padrao'] / cv_produto['preco_medio']
).fillna(0)

# Checar quantos grupos têm poucas transações (CV pouco confiável com poucas amostras)
print(f"\n======= Contagem de Quantidade de transações nos grupos de Produtos =======")
print(f"\nGrupos com 2+ transações: {len(cv_produto_valido)} de {len(cv_produto)}")
print(cv_produto['qtd_transacoes'].value_counts().sort_index().head(10))

# Selecionar SOMENTE os grupos com 2+ transações (CV mensurável de fato)
cv_produto_valido = cv_produto[cv_produto['qtd_transacoes'] >= 2]

# Distribuição geral
print(f"\n======= Distribuição nos Percentis =======\n")
print(cv_produto_valido['coeficiente_variacao'].quantile([0.25, 0.5, 0.75, 0.9, 0.95]))




======= Contagem de Quantidade de transações nos grupos de Produtos =======

Grupos com 2+ transações: 10087 de 14865
qtd_transacoes
1     4778
2     2050
3     1140
4      820
5      616
6      455
7      411
8      334
9      308
10     261
Name: count, dtype: int64

======= Distribuição nos Percentis =======

0.25    0.231276
0.50    0.426432
0.75    0.718390
0.90    1.033573
0.95    1.259780
Name: coeficiente_variacao, dtype: float64




=================== COEFICIENTE DE VARIAÇÃO ===================

CONCLUSÃO E INSIGHTS GERADOS

Grupos com apenas 1 transação serão classificados separadamente como *Não Aplicável*,<br/> 
por não possuírem variação mensurável. 

Para os 10.087 grupos com 2 ou mais transações, os limiares de classificação foram definidos <br/> 
pelos quartis da distribuição do CV: 

- Baixa (CV ≤ 0,23, até o 1º quartil)
- Média (CV entre 0,23 e 0,72, entre 1º e 3º quartil)
- Alta (CV > 0,72, acima do 3º quartil). 

Produtos com Alta dispersão indicam maior variabilidade de preço entre transações <br/> 
e representam prioridade para investigação de mercado.

***Usar quartis (p25/p75) é uma escolha metodológica objetiva e comum para esse tipo de classificação e <br/>
divide os produtos em blocos comparáveis de tamanho similar (25%/50%/25%), sem arbitrariedade.***

====================================================================


In [571]:

########################################################################
# CRIAÇÃO DAS COLUNAS: coeficiente_variacao e faixa_dispersao_preco
# (calculadas por codigo_br + unidade_fornecimento_capacidade, 
# usando apenas transações válidas, e depois propagadas para todo o df_ouro)
########################################################################

# 1. Base de cálculo: apenas transações válidas
df_validos = df_ouro[df_ouro['flag_qualidade_dado'] == 'Válido']

cv_produto = df_validos.groupby(['codigo_br', 'unidade_fornecimento_capacidade'])['preco_unitario'].agg(
    preco_medio='mean',
    preco_desvio_padrao=lambda x: x.std(ddof=0),
    qtd_transacoes='count'
).reset_index()

cv_produto['coeficiente_variacao'] = (
    cv_produto['preco_desvio_padrao'] / cv_produto['preco_medio']
).fillna(0)

# 2. Classificação em faixas utilizando CV encontrado nos quartis
def classificar_dispersao(row):
    if row['qtd_transacoes'] < 2:
        return 'Não Aplicável'
    elif row['coeficiente_variacao'] <= 0.23:
        return 'Baixa'
    elif row['coeficiente_variacao'] <= 0.72:
        return 'Média'
    else:
        return 'Alta'

cv_produto['faixa_dispersao_preco'] = cv_produto.apply(classificar_dispersao, axis=1)

# 3. Merge de volta para df_ouro (todas as transações recebem a classificação do seu grupo)
df_ouro = df_ouro.merge(
    cv_produto[['codigo_br', 'unidade_fornecimento_capacidade', 'coeficiente_variacao', 'faixa_dispersao_preco']],
    on=['codigo_br', 'unidade_fornecimento_capacidade'],
    how='left'
)

# Ajuste de tipo
df_ouro['faixa_dispersao_preco'] = df_ouro['faixa_dispersao_preco'].astype('category')

# preenchimento de 5 registros 'suspeitos' que ficaram sem classificação
df_ouro['faixa_dispersao_preco'] = df_ouro['faixa_dispersao_preco'].fillna('Não Aplicável')

# 4. Conferência
print(f"\n======= Contagem das Faixas de Dispersão no Preço no df_ouro =======\n")
print(df_ouro['faixa_dispersao_preco'].value_counts())

print(f"\n======= Camada Ouro (Tabela Única - segunda versão) gerada com sucesso! =======")
print(f"Shape do df_ouro: {df_ouro.shape[0]} linhas x {df_ouro.shape[1]} colunas.")





======= Contagem das Faixas de Dispersão no Preço no df_ouro =======

faixa_dispersao_preco
Média            190447
Alta             129229
Baixa             18113
Não Aplicável      4908
Name: count, dtype: int64

======= Camada Ouro (Tabela Única - segunda versão) gerada com sucesso! =======
Shape do df_ouro: 342697 linhas x 29 colunas.



```
┌───────────────────────────────────────────────────────────────────────┐
│       2.2.4 -> Criação de Star Schema                                 │
│                (Tabela Fato e Dimensões)                              │
└───────────────────────────────────────────────────────────────────────┘
```

In [572]:
###############################################################
# AUDITORIA DE DUPLICIDADES EM INSTITUIÇÕES (CNPJ)
###############################################################

# Antes da criação da dim_instituição, verificar duplicidades
col_audit = ['cnpj_instituicao', 'nome_instituicao', 'esfera', 'municipio_instituicao', 'uf']

# Identificar CNPJs que aparecem em mais de uma combinação de atributos
cnpjs_duplicados = (
    df_ouro[col_audit]
    .drop_duplicates()['cnpj_instituicao']
    .value_counts()
)
cnpjs_duplicados = cnpjs_duplicados[cnpjs_duplicados > 1].index

df_investigacao = (
    df_ouro[df_ouro['cnpj_instituicao'].isin(cnpjs_duplicados)][col_audit]
    .drop_duplicates()
    .sort_values(by='cnpj_instituicao')
)

print(f"\n======= Total de CNPJs Instituição com variações de cadastro: {len(cnpjs_duplicados)} =======")
if len(cnpjs_duplicados) > 0:
    display(df_investigacao)


======= Total de CNPJs Instituição com variações de cadastro: 9 =======


,cnpj_instituicao,nome_instituicao,esfera,municipio_instituicao,uf
26011,09.307.789/0001-00,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,RANCHO ALEGRE D`OESTE,PR
300025,09.307.789/0001-00,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,RANCHO ALEGRE D'OESTE,PR
211364,11.295.659/0001-39,FUNDO MUNICIPAL DE SAUDE DE SAO FELIPE D`OESTE,MUNICIPAL,SAO FELIPE D`OESTE,RO
287487,11.295.659/0001-39,FUNDO MUNICIPAL DE SAUDE DE SAO FELIPE D'OESTE,MUNICIPAL,SAO FELIPE D'OESTE,RO
16736,11.402.806/0001-22,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,ALTA FLORESTA D`OESTE,RO
323564,11.402.806/0001-22,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,ALTA FLORESTA D'OESTE,RO
41588,11.471.451/0001-23,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,OLHO D`AGUA,PB
328916,11.471.451/0001-23,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,OLHO D'AGUA,PB
43434,11.811.613/0001-25,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,SANTA LUZIA D`OESTE,RO
263491,11.811.613/0001-25,FUNDO MUNICIPAL DE SAUDE,MUNICIPAL,SANTA LUZIA D'OESTE,RO


In [573]:
###############################################################
# AUDITORIA DE DUPLICIDADES EM FORNECEDORES (CNPJ)
###############################################################

# Antes da criação da dim_fornecedor, verificar duplicidades
col_audit = ['cnpj_fornecedor', 'fornecedor']

# Identificar CNPJs que aparecem em mais de uma combinação de atributos
cnpjs_duplicados = (
    df_ouro[col_audit]
    .drop_duplicates()['cnpj_fornecedor']
    .value_counts()
)
cnpjs_duplicados = cnpjs_duplicados[cnpjs_duplicados > 1].index

df_investigacao = (
    df_ouro[df_ouro['cnpj_fornecedor'].isin(cnpjs_duplicados)][col_audit]
    .drop_duplicates()
    .sort_values(by='cnpj_fornecedor')
)

print(f"\n======= Total de CNPJs Fornecedor com variações de cadastro: {len(cnpjs_duplicados)} =======")
if len(cnpjs_duplicados) > 0:
    display(df_investigacao)


======= Total de CNPJs Fornecedor com variações de cadastro: 0 =======


In [574]:
###############################################################
# AUDITORIA DE DUPLICIDADES EM FABRICANTES (CNPJ)
###############################################################

# Antes da criação da dim_fabricante, verificar duplicidades
col_audit = ['cnpj_fabricante', 'fabricante']

# Identificar CNPJs que aparecem em mais de uma combinação de atributos
cnpjs_duplicados = (
    df_ouro[col_audit]
    .drop_duplicates()['cnpj_fabricante']
    .value_counts()
)
cnpjs_duplicados = cnpjs_duplicados[cnpjs_duplicados > 1].index

df_investigacao = (
    df_ouro[df_ouro['cnpj_fabricante'].isin(cnpjs_duplicados)][col_audit]
    .drop_duplicates()
    .sort_values(by='cnpj_fabricante')
)

print(f"\n======= Total de CNPJs Fabricante com variações de cadastro: {len(cnpjs_duplicados)} =======")
if len(cnpjs_duplicados) > 0:
    display(df_investigacao)


======= Total de CNPJs Fabricante com variações de cadastro: 0 =======


In [575]:
##################################################################################
# CAMADA OURO - ABORDAGEM 2: STAR SCHEMA (FATO + DIMENSÕES)
# fato_compras / dim_calendario / dim_instituicao / dim_fornecedor / dim_fabricante / dim_produto
##################################################################################

####################################################
# 1. DIMENSÃO CALENDÁRIO (dim_calendario)
####################################################

# armazenar a menor e maior data dos registros
data_min = df_ouro['compra'].min()
data_max = df_ouro['compra'].max()

# criar um 'DatetimeIndex' com todos os dias entre a data mínima e máxima
datas = pd.date_range(start=data_min, end=data_max, freq='D')

# criar o dataframe com a primeira coluna
dim_calendario = pd.DataFrame({'dt_completa': datas})

# criar a Surrogate Key (usada como Primary Key da tabela)
dim_calendario['sk_calendario'] = dim_calendario['dt_completa'].dt.strftime('%Y%m%d').astype('int32')

# criação das próximas colunas
dim_calendario['nr_ano'] = dim_calendario['dt_completa'].dt.year.astype('int16')
dim_calendario['nr_mes'] = dim_calendario['dt_completa'].dt.month.astype('int8')

# Mapeamento para Mês e Dia da Semana
mapa_meses = {
    1: 'JAN', 2: 'FEV', 3: 'MAR', 4: 'ABR', 5: 'MAI', 6: 'JUN',
    7: 'JUL', 8: 'AGO', 9: 'SET', 10: 'OUT', 11: 'NOV', 12: 'DEZ'
}
mapa_dias = {
    0: 'SEG', 1: 'TER', 2: 'QUA', 3: 'QUI',
    4: 'SEX', 5: 'SAB', 6: 'DOM'
}

# criação das próximas colunas
dim_calendario['nm_mes'] = dim_calendario['nr_mes'].map(mapa_meses).astype('category')
dim_calendario['ds_ano_mes'] = dim_calendario['dt_completa'].dt.strftime('%Y%m').astype('string')
dim_calendario['ds_periodo'] = dim_calendario['dt_completa'].dt.strftime('%m/%Y').astype('string')
dim_calendario['nr_trimestre'] = dim_calendario['dt_completa'].dt.quarter.astype('int8')
dim_calendario['nr_dia_semana'] = dim_calendario['dt_completa'].dt.dayofweek.astype('int8')
dim_calendario['nm_dia_semana'] = dim_calendario['nr_dia_semana'].map(mapa_dias).astype('category')

####################################################
# 2. DIMENSÃO INSTITUIÇÃO (dim_instituicao) - Desduplicada por CNPJ
####################################################

# selecionar as colunas
cols_i = ['nome_instituicao', 'esfera', 'municipio_instituicao', 'uf']
df_ouro = df_ouro.sort_values('compra')

# criar o dataframe agrupando por cnpj e selecionando o último (mais recente)
dim_instituicao = (
    df_ouro.groupby('cnpj_instituicao')[cols_i]
    .last()  # Seleciona o último registro atualizado para cada CNPJ
    .reset_index()
)

# criar a Surrogate Key (usada como Primary Key da tabela)
dim_instituicao['sk_instituicao'] = (dim_instituicao.index + 1).astype('int32')

# Reordena com a PK no início
dim_instituicao = dim_instituicao[['sk_instituicao', 'cnpj_instituicao'] + cols_i]

####################################################
# 3. DIMENSÃO PRODUTO (dim_produto)
####################################################

col_p = [
    'codigo_br', 'descricao_catmat', 'categoria_insumo'
]
dim_produto = df_ouro[col_p].drop_duplicates().reset_index(drop=True)
dim_produto['sk_produto'] = (dim_produto.index + 1).astype('int32')

dim_produto = dim_produto[['sk_produto'] + col_p]

####################################################
# 4. DIMENSÃO FORNECEDOR (dim_fornecedor)
####################################################

dim_fornecedor = (
    df_ouro.groupby('cnpj_fornecedor')['fornecedor']
    .last()
    .reset_index()
)
dim_fornecedor['sk_fornecedor'] = (dim_fornecedor.index + 1).astype('int32')
dim_fornecedor = dim_fornecedor[['sk_fornecedor', 'cnpj_fornecedor', 'fornecedor']]

####################################################
# 5. DIMENSÃO FABRICANTE (dim_fabricante)
####################################################

dim_fabricante = (
    df_ouro.groupby('cnpj_fabricante')['fabricante']
    .last()
    .reset_index()
)
dim_fabricante['sk_fabricante'] = (dim_fabricante.index + 1).astype('int32')
dim_fabricante = dim_fabricante[['sk_fabricante', 'cnpj_fabricante', 'fabricante']]

####################################################
# 6. TABELA FATO COMPRAS (fato_compras)
####################################################

# criar dataframe temporário
df_fato_temp = df_ouro.copy()

# mapeamento das chaves de data (sk_calendario)
df_fato_temp['sk_calendario'] = df_fato_temp['compra'].dt.strftime('%Y%m%d').astype('int32')

# merges para resgatar as SKs das dimensões
# a separação das [sk] + col no merge serve para não repetir as colunas, adicionando somente a sk
# 'on=col' irá comparar toda a lista de colunas 
# 'how=left' preserva todas linhas da fato mesmo que não exista correspondência da dimensão
df_fato_temp = df_fato_temp.merge(dim_instituicao[['sk_instituicao', 'cnpj_instituicao']], on='cnpj_instituicao', how='left')
df_fato_temp = df_fato_temp.merge(dim_produto[['sk_produto'] + col_p], on=col_p, how='left')
df_fato_temp = df_fato_temp.merge(dim_fornecedor[['sk_fornecedor', 'cnpj_fornecedor']], on='cnpj_fornecedor', how='left')
df_fato_temp = df_fato_temp.merge(dim_fabricante[['sk_fabricante', 'cnpj_fabricante']], on='cnpj_fabricante', how='left')


# selecionar as colunas da Fato (Apenas FKs, Atributos de Contexto e Métricas)
col_fato = [
    'sk_calendario', 'sk_instituicao', 'sk_produto', 'sk_fornecedor', 'sk_fabricante',
    'modalidade_compra', 'tipo_compra', 
    'unidade_fornecimento', 'unidade_fornecimento_capacidade', 'generico', 'anvisa',
    'flag_qualidade_dado', 'coeficiente_variacao', 'faixa_dispersao_preco',
    'qtd_itens_comprados', 'preco_unitario', 'preco_total'
]
fato_compras = df_fato_temp[col_fato].copy()

print("Star Schema gerado com sucesso!")
print(f"-> dim_calendario:  {dim_calendario.shape[0]:,} linhas x {dim_calendario.shape[1]} colunas")
print(f"-> dim_instituicao: {dim_instituicao.shape[0]:,} linhas x {dim_instituicao.shape[1]} colunas")
print(f"-> dim_produto:     {dim_produto.shape[0]:,} linhas x {dim_produto.shape[1]} colunas")
print(f"-> dim_fornecedor:  {dim_fornecedor.shape[0]:,} linhas x {dim_fornecedor.shape[1]} colunas")
print(f"-> dim_fabricante:  {dim_fabricante.shape[0]:,} linhas x {dim_fabricante.shape[1]} colunas")
print(f"-> fato_compras:    {fato_compras.shape[0]:,} linhas x {fato_compras.shape[1]} colunas")

Star Schema gerado com sucesso!
-> dim_calendario:  2,256 linhas x 10 colunas
-> dim_instituicao: 831 linhas x 6 colunas
-> dim_produto:     12,994 linhas x 4 colunas
-> dim_fornecedor:  3,502 linhas x 3 colunas
-> dim_fabricante:  2,290 linhas x 3 colunas
-> fato_compras:    342,697 linhas x 17 colunas


In [576]:

# Auditoria de integridade: Verificar se algum SK da tabela fato não foi cadastrado

print("\n======= CONTAGEM DE SK's NULOS NA TABELA FATO =======\n")
# Contar quantos NaN existem em cada SK
print(fato_compras[['sk_calendario', 'sk_instituicao', 'sk_produto', 'sk_fornecedor', 'sk_fabricante']].isna().sum())



======= CONTAGEM DE SK's NULOS NA TABELA FATO =======

sk_calendario     0
sk_instituicao    0
sk_produto        0
sk_fornecedor     0
sk_fabricante     0
dtype: int64


---

```
┌───────────────────────────────────────────────────────────────────────┐
│ 3- LOAD - Exportar DataFrames para novas bases (arquivos)             │
│    (Para persistência dos dados tratados e consumo do BI)             │
└───────────────────────────────────────────────────────────────────────┘
```

In [577]:

# Criar pastas de destino caso não existam
os.makedirs('dados/prata', exist_ok=True)
os.makedirs('dados/ouro/flat', exist_ok=True)
os.makedirs('dados/ouro/star_schema', exist_ok=True)

# conforme solicitado no briefing do projeto a base de dados 
# concatenada deve ser salva com nome 'BPS_20_26_NomeDoAluno.csv'

# os arquivos foram salvos em 3 tipos de extensão 
# '.csv', '.csv.gz' e '.parquet'
# para fins de estudo em relação ao tamanho final de cada formato

############################
# SALVAR CAMADA PRATA
############################

print("\nSalvando Camada Prata...")
df_prata.to_csv('dados/prata/BPS_20_26_FelipeVampre_prata.csv', index=False, encoding='utf-8')
df_prata.to_csv('dados/prata/BPS_20_26_FelipeVampre_prata.csv.gz', index=False, compression='gzip', encoding='utf-8')
df_prata.to_parquet('dados/prata/BPS_20_26_FelipeVampre_prata.parquet', index=False)

#######################################################################
# SALVAR CAMADA OURO - ABORDAGEM 1 (Tabela Única / Flat Table)
#######################################################################

# como a camada ouro é a tabela final de consumo do BI
# o nome foi salvo sem o prefixo '_ouro'

print("Salvando Camada Ouro (Flat Table)...")
df_ouro.to_csv('dados/ouro/flat/BPS_20_26_FelipeVampre.csv', index=False, encoding='utf-8')
df_ouro.to_csv('dados/ouro/flat/BPS_20_26_FelipeVampre.csv.gz', index=False, compression='gzip', encoding='utf-8')
df_ouro.to_parquet('dados/ouro/flat/BPS_20_26_FelipeVampre.parquet', index=False)

#######################################################################
# SALVAR CAMADA OURO - ABORDAGEM 2 (Star Schema)
#######################################################################

print("Salvando Camada Ouro (Star Schema)...")
tabelas_star = {
    'fato_compras': fato_compras,
    'dim_calendario': dim_calendario,
    'dim_instituicao': dim_instituicao,
    'dim_produto': dim_produto,
    'dim_fornecedor': dim_fornecedor,
    'dim_fabricante' : dim_fabricante
}

for nome_tabela, df_tabela in tabelas_star.items():
    caminho_base = f'dados/ouro/star_schema/{nome_tabela}'
    df_tabela.to_csv(f'{caminho_base}.csv', index=False, encoding='utf-8')
    df_tabela.to_csv(f'{caminho_base}.csv.gz', index=False, compression='gzip', encoding='utf-8')
    df_tabela.to_parquet(f'{caminho_base}.parquet', index=False)

print("\nTodos os arquivos da Camada Prata e Ouro foram salvos com sucesso!")



Salvando Camada Prata...
Salvando Camada Ouro (Flat Table)...
Salvando Camada Ouro (Star Schema)...

Todos os arquivos da Camada Prata e Ouro foram salvos com sucesso!
